# 🏆 Golden Dataset Generator
**6-Step Pipeline for generating evaluation ground truth from parent chunks**

1. Document Ingestion — Load parent chunks with metadata
2. Stratified Context Selection — Sample diverse chunks across chapters
3. Evolutionary Query Generation — Generate Simple, Reasoning, Multi-context questions
4. Ground Truth Synthesis — Generate answers with evidence sentences
5. Quality Filtering — Two-pass LLM-as-judge filter
6. Dataset Formatting — Export rich JSONL dataset


## Step 0: Install Dependencies & Imports


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
!pip install langchain langchain-core langchain-community langchain-nvidia-ai-endpoints langchain-huggingface langchain-chroma chromadb sentence-transformers


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 108.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.8/61.8 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 86.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 101.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 73.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 97.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.

In [2]:

import os
import re
import json
import random
import hashlib
from collections import defaultdict
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.messages import HumanMessage
from langchain_chroma import Chroma
from langchain_classic.storage import LocalFileStore, create_kv_docstore
import getpass




In [5]:
!ls drive

MyDrive


In [7]:
current_dir = os.getcwd()

In [8]:
current_dir

'/content'

In [9]:
current_dir = f"{current_dir}/drive/MyDrive/acm/ACM_Hackathon_Project"
current_dir

'/content/drive/MyDrive/acm/ACM_Hackathon_Project'

## Step 1: Document Ingestion — Load Parent Chunks


In [10]:
if "NVIDIA_API_KEY" in os.environ:
    del os.environ["NVIDIA_API_KEY"]
if "NVIDIA_API_KEY" not in os.environ:
    os.environ["NVIDIA_API_KEY"] = getpass.getpass("Enter your NVIDIA NIM API Key: ")
# === Load the embedding model (same as your pipeline) ===
local_embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={'device': 'cuda'},
    encode_kwargs={'normalize_embeddings': True}
)

# === Load VectorStore and Parent Store ===
persist_dir = f"{current_dir}/chroma_db_local/chroma_db_local"  # Adjust path
parent_store_dir = f"{current_dir}/parent_store_local/parent_store_local"  # Adjust path

vectorstore = Chroma(
    collection_name="med_rag_local",
    embedding_function=local_embeddings,
    persist_directory=persist_dir,
)

fs = LocalFileStore(parent_store_dir)
store = create_kv_docstore(fs)

# === Load LLM ===
generator_llm = ChatNVIDIA(
    model="meta/llama-3.3-70b-instruct",
    api_key=os.environ.get("NVIDIA_API_KEY"),
    temperature=0.7,
    max_completion_tokens=2048,
)

judge_llm = ChatNVIDIA(
    model="meta/llama-3.3-70b-instruct",
    api_key=os.environ.get("NVIDIA_API_KEY"),
    temperature=0.0,  # Deterministic for judging
    max_completion_tokens=1024,
)

print("✅ Models and stores loaded")


Enter your NVIDIA NIM API Key: ··········


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

✅ Models and stores loaded


### Load all parent chunks from the vectorstore metadata
We retrieve parent documents from the store to use as grounding context.


In [13]:
from concurrent.futures import ThreadPoolExecutor, as_completed

# 🔹 Step 1: Get all child chunks
all_child_docs = vectorstore.get(include=["metadatas"])

# 🔹 Step 2: Extract unique parent IDs (FASTER)
parent_metadata_map = {}
parent_ids = list({
    meta["doc_id"]: meta
    for meta in all_child_docs["metadatas"]
    if meta.get("doc_id")
}.keys())

# Build metadata map properly
for meta in all_child_docs["metadatas"]:
    pid = meta.get("doc_id")
    if pid and pid not in parent_metadata_map:
        parent_metadata_map[pid] = meta

print(f"📄 Found {len(parent_ids)} unique parent chunks")


# 🔹 Step 3: Create batches
BATCH_SIZE = 200
MAX_WORKERS = 6

batches = [
    parent_ids[i:i + BATCH_SIZE]
    for i in range(0, len(parent_ids), BATCH_SIZE)
]


# 🔹 Step 4: Define batch processor
def process_batch(batch_ids):
    batch_results = fs.mget(batch_ids)
    batch_chunks = []

    for pid, parent_bytes in zip(batch_ids, batch_results):
        if parent_bytes:
            text = parent_bytes.decode("utf-8") if isinstance(parent_bytes, bytes) else str(parent_bytes)
            meta = parent_metadata_map.get(pid, {})

            batch_chunks.append({
                "parent_id": pid,
                "text": text,
                "chapter": meta.get("chapter", "Unknown"),
                "section": meta.get("section", "Unknown"),
                "section_title": meta.get("section-title", "Unknown"),
                "page_start": meta.get("page_start", None),
                "page_end": meta.get("page_end", None),
                "has_image": bool("![" in text),
                "char_length": len(text),
            })

    return batch_chunks


# 🔹 Step 5: Run in parallel (SAFE variable)
parent_chunks = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [executor.submit(process_batch, batch) for batch in batches]

    for i, future in enumerate(as_completed(futures)):
        result = future.result()
        parent_chunks.extend(result)

        print(f"✅ Completed batch {i+1}/{len(batches)} | Total loaded: {len(parent_chunks)}")


# 🔹 Step 6: Final summary
print(f"\n🚀 Loaded {len(parent_chunks)} parent chunks (parallel version)")


# 🔹 Step 7: Preview
for p in parent_chunks[:3]:
    print(
        f"[{p['section']}] {p['chapter']} — {p['section_title']} "
        f"(pages {p['page_start']}-{p['page_end']}, {p['char_length']} chars)"
    )

📄 Found 1100 unique parent chunks
✅ Completed batch 1/6 | Total loaded: 100
✅ Completed batch 2/6 | Total loaded: 300
✅ Completed batch 3/6 | Total loaded: 500
✅ Completed batch 4/6 | Total loaded: 700
✅ Completed batch 5/6 | Total loaded: 900
✅ Completed batch 6/6 | Total loaded: 1100

🚀 Loaded 1100 parent chunks (parallel version)
[15.11] Psychological Disorders — Personality Disorders (pages 594-594, 323 chars)
[15.9] Psychological Disorders — Dissociative Disorders (pages 594-594, 2415 chars)
[15.9] Psychological Disorders — Dissociative Disorders (pages 594-594, 1432 chars)


## Step 2: Stratified Context Selection
Sample diverse chunks across chapters, ensuring coverage of images, tables, and varied lengths.


In [14]:

def stratified_sample(parent_chunks, target_count=80):
    """
    Sample chunks ensuring:
    - Equal representation across chapters
    - Mix of short/long chunks
    - Include chunks with images
    - Skip very short chunks (< 200 chars, likely headers only)
    """
    # Filter out very short chunks
    viable = [c for c in parent_chunks if c["char_length"] >= 200]
    print(f"Viable chunks (>200 chars): {len(viable)} / {len(parent_chunks)}")

    # Group by chapter
    by_chapter = defaultdict(list)
    for chunk in viable:
        by_chapter[chunk["chapter"]].append(chunk)

    # Calculate per-chapter quota
    num_chapters = len(by_chapter)
    per_chapter = max(2, target_count // num_chapters)
    print(f"Sampling ~{per_chapter} chunks from each of {num_chapters} chapters")

    sampled = []
    for chapter, chunks in sorted(by_chapter.items()):
        # Sort by length to get diversity
        chunks_sorted = sorted(chunks, key=lambda x: x["char_length"])

        # Pick from different length ranges
        n = min(per_chapter, len(chunks))
        if n >= 3:
            # Take 1 short, some medium, 1 long
            indices = [0]  # shortest
            indices.append(len(chunks_sorted) - 1)  # longest
            # Fill rest from middle
            middle = chunks_sorted[1:-1]
            random.shuffle(middle)
            indices_mid = list(range(1, min(n - 2 + 1, len(chunks_sorted) - 1)))
            random.shuffle(indices_mid)
            indices.extend(indices_mid[:n - 2])
        else:
            indices = list(range(n))

        for idx in indices:
            if idx < len(chunks_sorted):
                sampled.append(chunks_sorted[idx])

        print(f"  📖 {chapter}: sampled {min(n, len(indices))}/{len(chunks)} chunks")

    # Ensure we have some image-containing chunks
    image_chunks = [c for c in viable if c["has_image"] and c not in sampled]
    if image_chunks:
        extra = random.sample(image_chunks, min(5, len(image_chunks)))
        sampled.extend(extra)
        print(f"  🖼️ Added {len(extra)} extra image-containing chunks")

    # Trim to target
    if len(sampled) > target_count:
        sampled = random.sample(sampled, target_count)

    print(f"\n✅ Final sample: {len(sampled)} chunks")
    return sampled


sampled_chunks = stratified_sample(parent_chunks, target_count=80)


Viable chunks (>200 chars): 1100 / 1100
Sampling ~4 chunks from each of 17 chapters
  📖 Biopsychology: sampled 4/10 chunks
  📖 Emotion and Motivation: sampled 4/79 chunks
  📖 Industrial-Organizational Psychology: sampled 4/58 chunks
  📖 Introduction to Psychology: sampled 4/50 chunks
  📖 Learning: sampled 4/51 chunks
  📖 Lifespan Development: sampled 4/76 chunks
  📖 Memory: sampled 4/57 chunks
  📖 Personality: sampled 4/98 chunks
  📖 Psychological Disorders: sampled 4/176 chunks
  📖 Psychological Research: sampled 4/158 chunks
  📖 Sensation and Perception: sampled 4/29 chunks
  📖 Social Psychology: sampled 4/32 chunks
  📖 States of Consciousness: sampled 4/46 chunks
  📖 Stress, Lifestyle, and Health: sampled 4/69 chunks
  📖 Therapy and Treatment: sampled 4/10 chunks
  📖 Thinking and Intelligence: sampled 4/62 chunks
  📖 Unknown: sampled 4/39 chunks
  🖼️ Added 5 extra image-containing chunks

✅ Final sample: 73 chunks


## Step 3: Evolutionary Query Generation
Generate 3 types: Simple, Reasoning, Multi-context + Unanswerable


In [15]:

def generate_simple_question(chunk, llm):
    """Generate a straightforward factual question from a single chunk."""
    prompt = f"""Based on the following textbook passage, generate ONE factual question
that can be directly answered from this text. The question should require a specific,
detailed answer (not just yes/no).

Passage:
{chunk['text'][:2000]}

Generate a JSON response:
{{
    "question": "Your factual question here",
    "question_type": "simple"
}}

Respond with ONLY the JSON, no other text."""

    try:
        response = llm.invoke([HumanMessage(content=prompt)])
        return json.loads(response.content)
    except Exception as e:
        print(f"  ⚠️ Error generating simple question: {e}")
        return None


def generate_reasoning_question(chunk, llm):
    """Generate a question requiring analysis/reasoning about the content."""
    prompt = f"""Based on the following textbook passage, generate ONE question that
requires REASONING or ANALYSIS to answer. The question should ask about causes, effects,
comparisons, or explanations — not just direct recall.

Examples of reasoning questions:
- "How does X affect Y?"
- "Why is X considered more effective than Y?"
- "What would happen if X were removed from the process?"

Passage:
{chunk['text'][:2000]}

Generate a JSON response:
{{
    "question": "Your reasoning question here",
    "question_type": "reasoning"
}}

Respond with ONLY the JSON, no other text."""

    try:
        response = llm.invoke([HumanMessage(content=prompt)])
        return json.loads(response.content)
    except Exception as e:
        print(f"  ⚠️ Error generating reasoning question: {e}")
        return None


def generate_multi_context_question(chunk1, chunk2, llm):
    """Generate a question requiring info from TWO different chunks."""
    prompt = f"""Based on the following TWO textbook passages, generate ONE question that
requires information from BOTH passages to answer fully. The question should bridge
concepts across the two passages.

Passage 1 (Section {chunk1['section']} — {chunk1['section_title']}):
{chunk1['text'][:1500]}

Passage 2 (Section {chunk2['section']} — {chunk2['section_title']}):
{chunk2['text'][:1500]}

Generate a JSON response:
{{
    "question": "Your multi-context question here",
    "question_type": "multi_context"
}}

Respond with ONLY the JSON, no other text."""

    try:
        response = llm.invoke([HumanMessage(content=prompt)])
        return json.loads(response.content)
    except Exception as e:
        print(f"  ⚠️ Error generating multi-context question: {e}")
        return None


def generate_unanswerable_question(chunk, llm):
    """Generate a question that sounds related but CANNOT be answered from the text."""
    prompt = f"""Based on this textbook passage about psychology, generate ONE question that:
1. Sounds plausible and related to the topic
2. CANNOT be answered from the given text
3. Would require information NOT present in this passage

Passage:
{chunk['text'][:1500]}

Generate a JSON response:
{{
    "question": "Your unanswerable question here",
    "question_type": "unanswerable"
}}

Respond with ONLY the JSON, no other text."""

    try:
        response = llm.invoke([HumanMessage(content=prompt)])
        return json.loads(response.content)
    except Exception as e:
        print(f"  ⚠️ Error generating unanswerable question: {e}")
        return None


In [16]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import random

# === Setup ===
random.seed(42)
random.shuffle(sampled_chunks)

n_total = len(sampled_chunks)
n_simple = int(n_total * 0.40)
n_reasoning = int(n_total * 0.25)
n_multi = int(n_total * 0.20)
n_unanswerable = n_total - n_simple - n_reasoning - n_multi

print(f"🎯 Target: {n_simple} simple, {n_reasoning} reasoning, "
      f"{n_multi} multi-context, {n_unanswerable} unanswerable")

generated_pairs = []

MAX_WORKERS = 5  # keep low (LLM API rate limits)


# === TASK FUNCTIONS ===

def simple_task(i):
    chunk = sampled_chunks[i % len(sampled_chunks)]
    qa = generate_simple_question(chunk, generator_llm)
    if qa:
        qa["source_chunks"] = [chunk]
        qa["answerable"] = True
        qa["question_type"] = "simple"
    return qa, chunk, i


def reasoning_task(i):
    chunk = sampled_chunks[i % len(sampled_chunks)]
    qa = generate_reasoning_question(chunk, generator_llm)
    if qa:
        qa["source_chunks"] = [chunk]
        qa["answerable"] = True
        qa["question_type"] = "reasoning"
    return qa, chunk, i


def multi_task(i):
    c1 = sampled_chunks[i % len(sampled_chunks)]
    others = [c for c in sampled_chunks if c["chapter"] != c1["chapter"]]
    c2 = random.choice(others) if others else sampled_chunks[(i + 1) % len(sampled_chunks)]

    qa = generate_multi_context_question(c1, c2, generator_llm)
    if qa:
        qa["source_chunks"] = [c1, c2]
        qa["answerable"] = True
        qa["question_type"] = "multi"
    return qa, (c1, c2), i


def unanswerable_task(i):
    chunk = sampled_chunks[i % len(sampled_chunks)]
    qa = generate_unanswerable_question(chunk, generator_llm)
    if qa:
        qa["source_chunks"] = [chunk]
        qa["answerable"] = False
        qa["question_type"] = "unanswerable"
    return qa, chunk, i


# === RUN PARALLEL ===

def run_parallel(task_fn, count, label):
    print(f"\n📌 Generating {label} questions...")
    results = []

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [executor.submit(task_fn, i) for i in range(count)]

        for i, future in enumerate(as_completed(futures)):
            try:
                qa, chunk_info, idx = future.result()

                if qa:
                    generated_pairs.append(qa)
                    print(f"  [{i+1}/{count}] ✅")
                else:
                    print(f"  [{i+1}/{count}] ❌")

            except Exception as e:
                print(f"  [{i+1}/{count}] ⚠️ Error: {e}")

    return results


# === EXECUTION ===

run_parallel(simple_task, n_simple, "Simple")
run_parallel(reasoning_task, n_reasoning, "Reasoning")
run_parallel(multi_task, n_multi, "Multi-context")
run_parallel(unanswerable_task, n_unanswerable, "Unanswerable")


print(f"\n🚀 Generated {len(generated_pairs)} question-answer pairs")

🎯 Target: 29 simple, 18 reasoning, 14 multi-context, 12 unanswerable

📌 Generating Simple questions...
  [1/29] ✅
  [2/29] ✅
  [3/29] ✅
  [4/29] ✅
  [5/29] ✅
  [6/29] ✅
  [7/29] ✅
  [8/29] ✅
  [9/29] ✅
  [10/29] ✅
  [11/29] ✅
  [12/29] ✅
  [13/29] ✅
  [14/29] ✅
  ⚠️ Error generating simple question: [429] Too Many Requests
{'status': 429, 'title': 'Too Many Requests'}
  [15/29] ❌
  [16/29] ✅
  [17/29] ✅
  [18/29] ✅
  [19/29] ✅
  [20/29] ✅
  [21/29] ✅
  [22/29] ✅
  [23/29] ✅
  [24/29] ✅
  [25/29] ✅
  [26/29] ✅
  [27/29] ✅
  [28/29] ✅
  [29/29] ✅

📌 Generating Reasoning questions...
  [1/18] ✅
  [2/18] ✅
  [3/18] ✅
  [4/18] ✅
  [5/18] ✅
  ⚠️ Error generating reasoning question: [429] Too Many Requests
{'status': 429, 'title': 'Too Many Requests'}
  [6/18] ❌
  ⚠️ Error generating reasoning question: [429] Too Many Requests
{'status': 429, 'title': 'Too Many Requests'}
  [7/18] ❌
  [8/18] ✅
  [9/18] ✅
  [10/18] ✅
  [11/18] ✅
  [12/18] ✅
  ⚠️ Error generating reasoning question: [429] Too M

## Step 4: Ground Truth Synthesis
Generate "perfect" answers using ONLY the source chunks as context.


In [17]:

def generate_ground_truth(qa_pair, llm):
    """Generate a ground truth answer with evidence sentences."""

    source_texts = "\n\n---\n\n".join([c["text"][:2000] for c in qa_pair["source_chunks"]])

    if not qa_pair["answerable"]:
        # For unanswerable questions, ground truth is the refusal
        qa_pair["ground_truth"] = (
            "The provided textbook sections do not contain sufficient "
            "information to answer this question."
        )
        qa_pair["evidence_sentences"] = []
        return qa_pair

    prompt = f"""You are a textbook expert. Answer the following question using ONLY the
provided textbook passages. Your answer must be comprehensive and accurate.

Question: {qa_pair['question']}

Textbook Passages:
{source_texts}

Provide your response as JSON:
{{
    "ground_truth": "Your detailed, accurate answer based only on the passages above.",
    "evidence_sentences": [
        "Exact sentence or phrase from the passage that supports your answer.",
        "Another supporting sentence from the passage."
    ]
}}

Rules:
- Include 2-4 evidence sentences copied EXACTLY from the passages
- The answer must be FULLY supported by the evidence sentences
- Do NOT add any information not in the passages

Respond with ONLY the JSON, no other text."""

    try:
        response = llm.invoke([HumanMessage(content=prompt)])
        result = json.loads(response.content)
        qa_pair["ground_truth"] = result["ground_truth"]
        qa_pair["evidence_sentences"] = result.get("evidence_sentences", [])
        return qa_pair
    except Exception as e:
        print(f"  ⚠️ Error generating ground truth: {e}")
        qa_pair["ground_truth"] = None
        qa_pair["evidence_sentences"] = []
        return qa_pair


In [18]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from langchain_core.messages import HumanMessage

# 🔹 CONFIG
MAX_WORKERS = 3   # keep low to avoid rate limit
RETRIES = 2       # retry failed calls


# 🔹 FUNCTION: generate ground truth (NO JSON)
def generate_ground_truth(qa_pair, llm, retries=RETRIES):
    prompt = f"""
Answer the question based ONLY on the context.

Return ONLY the final answer.
No explanation.
No JSON.

Question: {qa_pair["question"]}
"""

    for _ in range(retries):
        try:
            response = llm.invoke([HumanMessage(content=prompt)])
            answer = response.content.strip()

            if answer:
                qa_pair["ground_truth"] = answer.replace("\n", " ").strip()
                return qa_pair

        except Exception as e:
            continue

    qa_pair["ground_truth"] = None
    return qa_pair


# 🔹 PARALLEL PROCESSING
def process_qa(qa):
    return generate_ground_truth(qa, generator_llm)


print("📝 Generating ground truth answers (parallel, stable)...")

results = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {
        executor.submit(process_qa, qa): i
        for i, qa in enumerate(generated_pairs)
    }

    for future in as_completed(futures):
        i = futures[future]
        qa = future.result()

        print(f"  [{i+1}/{len(generated_pairs)}] {qa['question_type']}: "
              f"{qa['question'][:60]}...", end=" ")

        if qa.get("ground_truth"):
            print("✅")
        else:
            print("❌")

        results.append(qa)


# 🔹 FILTER SUCCESSFUL ONES
generated_pairs = [qa for qa in results if qa.get("ground_truth")]

print(f"\n🚀 Ground truth generated for {len(generated_pairs)} pairs")

📝 Generating ground truth answers (parallel, stable)...
  [1/66] simple: What are the two types of personality tests mentioned in the... ✅
  [2/66] simple: What is the chapter title of the section that includes secti... ✅
  [3/66] simple: What is the title of the section in the chapter 'Stress, Lif... ✅
  [6/66] simple: What is the chapter title of the section where Prosocial Beh... ❌
  [4/66] simple: On which page does the section 6.3 about Operant Conditionin... ✅
  [7/66] simple: What is the title of the section where TABLE 6.2 is located?... ✅
  [9/66] simple: What is the title of the section in which the topic of death... ✅
  [5/66] simple: On what page does the section 4.3 Stages of Sleep start?... ✅
  [8/66] simple: What is the title of section 11.4 in the chapter on Personal... ✅
  [10/66] simple: What technique developed by Carl Rogers is known as client-c... ✅
  [11/66] simple: What is the title of the section in the chapter Introduction... ✅
  [12/66] simple: What part of th

## Step 5: Quality Filtering (Two-Pass LLM-as-Judge)


In [19]:
import json
import re
from langchain_core.messages import HumanMessage

def safe_parse_json(text):
    try:
        return json.loads(text)
    except:
        match = re.search(r'\{.*\}', text, re.DOTALL)
        if match:
            try:
                return json.loads(match.group())
            except:
                pass
    return {}


def quality_filter(qa_pair, judge_llm):
    """Stronger quality filter (balanced strictness)"""

    source_texts = "\n\n".join([c["text"][:1000] for c in qa_pair["source_chunks"]])

    # 🔒 HARD FILTERS (cheap + effective)
    if qa_pair.get("answerable", True):
        gt = qa_pair.get("ground_truth", "").strip().lower()

        if len(gt) < 20:
            return False
        if "i don't know" in gt or "not provided" in gt:
            return False


    # 🔹 UNANSWERABLE CASE
    if not qa_pair.get("answerable", True):

        prompt = f"""
Evaluate this question:

Question: {qa_pair['question']}

Score 1-5:
1. clarity
2. plausibility
3. unanswerable

Return ONLY JSON:
{{"clarity": N, "plausibility": N, "unanswerable": N}}
"""

        try:
            response = judge_llm.invoke([HumanMessage(content=prompt)])
            scores = safe_parse_json(response.content)
            qa_pair["quality_scores"] = scores

            # 🔥 slightly stricter
            return (
                scores.get("clarity", 0) >= 3 and
                scores.get("plausibility", 0) >= 4 and
                scores.get("unanswerable", 0) >= 4
            )

        except Exception:
            return False   # ❗ changed


    # 🔹 ANSWERABLE CASE
    prompt = f"""
Evaluate this QA pair:

Question: {qa_pair['question']}
Answer: {qa_pair['ground_truth']}

Context:
{source_texts[:1500]}

Score 1-5:
1. answerable
2. grounded
3. clarity
4. completeness

Return ONLY JSON:
{{"answerable": N, "grounded": N, "clarity": N, "completeness": N}}
"""

    try:
        response = judge_llm.invoke([HumanMessage(content=prompt)])
        scores = safe_parse_json(response.content)
        qa_pair["quality_scores"] = scores

        # 🔥 STRONGER FILTER (key change)
        passes = (
            scores.get("grounded", 0) >= 4 and      # ↑ most important
            scores.get("answerable", 0) >= 3 and
            scores.get("clarity", 0) >= 3 and
            scores.get("completeness", 0) >= 3
        )

        return passes

    except Exception as e:
        print(f"  ⚠️ Judge error: {e}")
        return False   # ❗ changed

In [20]:
from concurrent.futures import ThreadPoolExecutor, as_completed

print("🔍 Running quality filter (parallel)...")

filtered_pairs = []
rejected = 0

MAX_WORKERS = 5  # keep low (LLM rate limits)


def process_qa(i, qa):
    try:
        passes = quality_filter(qa, judge_llm)
        scores = qa.get("quality_scores", {})
        return i, qa, passes, scores
    except Exception as e:
        return i, qa, False, {"error": str(e)}


results = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [
        executor.submit(process_qa, i, qa)
        for i, qa in enumerate(generated_pairs)
    ]

    for future in as_completed(futures):
        i, qa, passes, scores = future.result()

        print(f"  [{i+1}/{len(generated_pairs)}] {qa['question'][:50]}...", end=" ")

        if passes:
            filtered_pairs.append(qa)
            print(f"✅ (scores: {scores})")
        else:
            rejected += 1
            print(f"❌ REJECTED (scores: {scores})")


print("\n📊 FINAL RESULTS")
print(f"✅ Passed: {len(filtered_pairs)}")
print(f"❌ Rejected: {rejected}")
print(f"📦 Final dataset size: {len(filtered_pairs)}")

🔍 Running quality filter (parallel)...
  [6/65] What is the title of the section in which the topi... ❌ REJECTED (scores: {})
  [5/65] What is the title of the section where TABLE 6.2 i... ❌ REJECTED (scores: {})
  [2/65] What is the chapter title of the section that incl... ❌ REJECTED (scores: {})
  [7/65] On what page does the section 4.3 Stages of Sleep ... ❌ REJECTED (scores: {})
  [4/65] On which page does the section 6.3 about Operant C... ❌ REJECTED (scores: {'answerable': 2, 'grounded': 2, 'clarity': 5, 'completeness': 5})
  [9/65] What technique developed by Carl Rogers is known a... ✅ (scores: {'answerable': 5, 'grounded': 5, 'clarity': 5, 'completeness': 5})
  [11/65] What part of the brain is the body's biological cl... ❌ REJECTED (scores: {})
  [8/65] What is the title of section 11.4 in the chapter o... ❌ REJECTED (scores: {'answerable': 1, 'grounded': 2, 'clarity': 5, 'completeness': 1})
  [3/65] What is the title of the section in the chapter 'S... ❌ REJECTED (scores: {

## Step 6: Dataset Formatting — Export Rich JSONL


In [21]:

def format_dataset(filtered_pairs):
    """Format into the final golden dataset schema."""
    dataset = []

    for i, qa in enumerate(filtered_pairs):
        entry = {
            "id": f"gt_{i+1:03d}",
            "question": qa["question"],
            "ground_truth": qa["ground_truth"],
            "evidence_sentences": qa.get("evidence_sentences", []),
            "golden_context_ids": [c["parent_id"] for c in qa["source_chunks"]],
            "relevant_sections": list(set(
                c["section"] for c in qa["source_chunks"]
            )),
            "relevant_pages": sorted(set(
                p for c in qa["source_chunks"]
                for p in range(
                    c.get("page_start") or 0,
                    (c.get("page_end") or 0) + 1
                )
                if p > 0
            )),
            "relevant_chapters": list(set(
                c["chapter"] for c in qa["source_chunks"]
            )),
            "question_type": qa["question_type"],
            "answerable": qa["answerable"],
            "difficulty": qa.get("quality_scores", {}).get("difficulty", None),
            "quality_scores": qa.get("quality_scores", {}),
        }
        dataset.append(entry)

    return dataset


golden_dataset = format_dataset(filtered_pairs)

# Save as JSONL
output_path = f"{current_dir}/docs/golden_dataset.jsonl"
with open(output_path, "w", encoding="utf-8") as f:
    for entry in golden_dataset:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

# Also save as regular JSON for easy reading
output_json = f"{current_dir}/docs/golden_dataset.json"
with open(output_json, "w", encoding="utf-8") as f:
    json.dump(golden_dataset, f, indent=2, ensure_ascii=False)

print(f"✅ Golden dataset saved!")
print(f"   📄 JSONL: {output_path}")
print(f"   📄 JSON:  {output_json}")
print(f"   📊 Total entries: {len(golden_dataset)}")


✅ Golden dataset saved!
   📄 JSONL: /content/drive/MyDrive/acm/ACM_Hackathon_Project/docs/golden_dataset.jsonl
   📄 JSON:  /content/drive/MyDrive/acm/ACM_Hackathon_Project/docs/golden_dataset.json
   📊 Total entries: 31


## Step 7: Dataset Statistics & Summary


In [22]:

from collections import Counter

types = Counter(e["question_type"] for e in golden_dataset)
answerable = sum(1 for e in golden_dataset if e["answerable"])
chapters = Counter(ch for e in golden_dataset for ch in e["relevant_chapters"])
difficulties = [e["difficulty"] for e in golden_dataset if e["difficulty"]]

print("=" * 60)
print("📊 GOLDEN DATASET SUMMARY")
print("=" * 60)
print(f"\nTotal entries: {len(golden_dataset)}")
print(f"Answerable:    {answerable}")
print(f"Unanswerable:  {len(golden_dataset) - answerable}")

print(f"\n📋 Question Type Distribution:")
for qtype, count in types.most_common():
    pct = count / len(golden_dataset) * 100
    bar = "█" * int(pct / 2)
    print(f"  {qtype:15s} {count:3d} ({pct:5.1f}%) {bar}")

print(f"\n📖 Chapter Coverage:")
for chapter, count in chapters.most_common():
    print(f"  {chapter}: {count} questions")

if difficulties:
    avg_diff = sum(difficulties) / len(difficulties)
    print(f"\n🎯 Average Difficulty: {avg_diff:.2f} / 5.0")

print("\n" + "=" * 60)


📊 GOLDEN DATASET SUMMARY

Total entries: 31
Answerable:    31
Unanswerable:  0

📋 Question Type Distribution:
  reasoning        13 ( 41.9%) ████████████████████
  multi            11 ( 35.5%) █████████████████
  simple            7 ( 22.6%) ███████████

📖 Chapter Coverage:
  States of Consciousness: 5 questions
  Personality: 5 questions
  Learning: 5 questions
  Therapy and Treatment: 4 questions
  Introduction to Psychology: 3 questions
  Sensation and Perception: 3 questions
  Psychological Disorders: 3 questions
  Psychological Research: 3 questions
  Emotion and Motivation: 2 questions
  Thinking and Intelligence: 2 questions
  Stress, Lifestyle, and Health: 2 questions
  Lifespan Development: 2 questions
  Memory: 1 questions
  Unknown: 1 questions
  Social Psychology: 1 questions



## Preview: Sample Entries


In [23]:

for entry in golden_dataset[:5]:
    print(f"\n{'─' * 60}")
    print(f"ID:       {entry['id']}")
    print(f"Type:     {entry['question_type']}")
    print(f"Question: {entry['question']}")
    print(f"Answer:   {entry['ground_truth'][:150]}...")
    print(f"Sections: {entry['relevant_sections']}")
    print(f"Pages:    {entry['relevant_pages']}")
    print(f"Scores:   {entry['quality_scores']}")



────────────────────────────────────────────────────────────
ID:       gt_001
Type:     simple
Question: What technique developed by Carl Rogers is known as client-centered or Rogerian therapy?
Answer:   Person-centered therapy...
Sections: ['16.2']
Pages:    [641]
Scores:   {'answerable': 5, 'grounded': 5, 'clarity': 5, 'completeness': 5}

────────────────────────────────────────────────────────────
ID:       gt_002
Type:     simple
Question: What is the title of the section in the chapter Introduction to Psychology where TABLE 1.1 is located?
Answer:   Subfields of Psychology...
Sections: ['1.4']
Pages:    [41]
Scores:   {'answerable': 5, 'grounded': 5, 'clarity': 5, 'completeness': 5}

────────────────────────────────────────────────────────────
ID:       gt_003
Type:     simple
Question: What is the title of the section that starts on page 173?
Answer:   The Context is Missing....
Sections: ['5.4']
Pages:    [173]
Scores:   {'answerable': 5, 'grounded': 5, 'clarity': 5, 'completen